# IBVAP — coarse detector training on Colab

13-class YOLO11s, leak-free 89k/10k split. Built for **Colab Pro** (Google AI Pro).

**Before you start:** upload `data/ibvap_coarse_dataset.tar` (~33 GB, built by
`scripts/pack_dataset_for_colab.py --tar`) to `MyDrive/ibvap/` in your Drive.

Runtime → Change runtime type → **A100** (or L4). Then Runtime → Run all.
Checkpoints sync to Drive every epoch, so a disconnect just means: reconnect,
run the setup cells, then run the **Resume** cell.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
!pip -q install 'ultralytics==8.4.140' 'torch' 2>/dev/null
import ultralytics, torch
print('ultralytics', ultralytics.__version__, '| cuda', torch.cuda.is_available(), torch.cuda.get_device_name(0))

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
DRIVE   = '/content/drive/MyDrive/ibvap'
TAR     = f'{DRIVE}/ibvap_coarse_dataset.tar'
DATA    = '/content/ibvap_data'          # fast local disk
RUNS    = f'{DRIVE}/runs'                # checkpoints live here (survive disconnects)
RUN     = 'ibvap_coarse'
os.makedirs(RUNS, exist_ok=True)
assert os.path.exists(TAR), f'not found: {TAR} — upload the tar to MyDrive/ibvap/ first'
print('tar size: %.1f GB' % (os.path.getsize(TAR)/1e9))

In [ ]:
# Extract once to local disk (~15–25 min). Skips if already done.
import os, subprocess, time
if not os.path.isdir(f'{DATA}/images'):
    os.makedirs(DATA, exist_ok=True)
    t = time.time()
    subprocess.run(['tar', '-xf', TAR, '-C', DATA], check=True)
    print('extracted in %.1f min' % ((time.time()-t)/60))
else:
    print('already extracted')
!ls {DATA} && head -c 200 {DATA}/dataset_coarse.yaml

In [ ]:
# Drive checkpoint sync — copies last.pt / best.pt / results.csv after every epoch
import shutil, os
from ultralytics import YOLO, settings
settings.update({'datasets_dir': DATA, 'runs_dir': '/content/runs'})

CKPT = f'{RUNS}/{RUN}'
os.makedirs(CKPT, exist_ok=True)

def _sync(trainer):
    try:
        w = trainer.save_dir / 'weights'
        for f in ('last.pt', 'best.pt'):
            if (w / f).exists():
                shutil.copy2(w / f, f'{CKPT}/{f}')
        if (trainer.save_dir / 'results.csv').exists():
            shutil.copy2(trainer.save_dir / 'results.csv', f'{CKPT}/results.csv')
    except Exception as e:
        print('sync warn:', e)

def attach(model):
    model.add_callback('on_fit_epoch_end', _sync)
    return model
print('checkpoint dir on Drive:', CKPT)

In [ ]:
# ---- TRAIN (fresh) ----
TRAIN_ARGS = dict(
    data=f'{DATA}/dataset_coarse.yaml',
    epochs=150, imgsz=640, batch=-1,          # A100: try batch=64 explicitly if -1 underfills
    patience=40, device=0, workers=8, cache='disk',
    optimizer='SGD', lr0=0.01, lrf=0.01, momentum=0.937, weight_decay=0.0005,
    warmup_epochs=3.0, cos_lr=True, close_mosaic=15,
    mixup=0.10, copy_paste=0.15, degrees=5.0, scale=0.5,
    project='/content/runs', name=RUN, exist_ok=True, plots=True,
)
model = attach(YOLO('yolo11s.pt'))
model.train(**TRAIN_ARGS)

In [ ]:
# ---- RESUME after a disconnect ----
# Run the setup cells above first (GPU, mount, extract-skip, sync).
import shutil, os
local_w = f'/content/runs/{RUN}/weights'
os.makedirs(local_w, exist_ok=True)
for f in ('last.pt', 'best.pt'):
    if os.path.exists(f'{RUNS}/{RUN}/{f}'):
        shutil.copy2(f'{RUNS}/{RUN}/{f}', f'{local_w}/{f}')
if os.path.exists(f'{RUNS}/{RUN}/results.csv'):
    shutil.copy2(f'{RUNS}/{RUN}/results.csv', f'/content/runs/{RUN}/results.csv')

model = attach(YOLO(f'{local_w}/last.pt'))
model.train(resume=True)

In [ ]:
# ---- Validate + save weights to Drive ----
import shutil
best = f'{RUNS}/{RUN}/best.pt'
m = YOLO(best)
metrics = m.val(data=f'{DATA}/dataset_coarse.yaml', imgsz=640, split='val')
print('mAP50', round(metrics.box.map50, 3), '| mAP50-95', round(metrics.box.map, 3))
shutil.copy2(best, f'{DRIVE}/ibvap_coarse_best.pt')
print('saved -> MyDrive/ibvap/ibvap_coarse_best.pt  (download this into backend/models/)')